# CreteValley CIM data import and profiles interactive powerflow simulation 

## Import Libraries

In [5]:
pip install openpyxl lxml


Note: you may need to restart the kernel to use updated packages.


In [14]:
import glob
import sys
import dpsim
import logging
import cimpy as cimpy
import math, cmath
import openpyxl
from villas.dataprocessing.readtools import *
print(dir(dpsim.GeneratorType)) # --> no PQnode available
print(dir(dpsim.LoadType)) # --> no PQnode available

['FullOrder', 'FullOrderVBR', 'IdealVoltageSource', 'NONE', 'PVNode', 'SG3OrderVBR', 'SG4OrderVBR', 'SG5OrderVBR', 'SG6aOrderVBR', 'SG6bOrderVBR', 'TransientStability', '__class__', '__delattr__', '__dir__', '__doc__', '__entries', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__index__', '__init__', '__init_subclass__', '__int__', '__le__', '__lt__', '__members__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '_pybind11_conduit_v1_', 'name', 'value']


AttributeError: module 'dpsim' has no attribute 'LoadType'

## Import CIM data

In [2]:
# Remove all existing handlers to prevent multiple logging configurations
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(filename='CIMimport.log', level=logging.INFO, filemode='w')

# filename = './Network_CIM_Data/Crete_equivalent_min_loading_activeOnly/'
filename = '/dpsim/examples/Notebooks/Grids/logs/Crete 2030/'
files = glob.glob(filename + '*.xml')
print(files)
import_result = cimpy.cim_import(files, "cgmes_v2_4_15")

with open('CIMimport.log', 'r') as file:
    for line in file:
        print(line.strip())

['/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___DL_.xml', '/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___GL_.xml', '/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___SSH_.xml', '/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___SV_.xml', '/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___TP_.xml', '/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z__EQ_.xml']
INFO:cimpy.cimimport:START of parsing file "/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___DL_.xml"
INFO:cimpy.cimimport:START of parsing file "/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___GL_.xml"
INFO:cimpy.cimimport:START of parsing file "/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___SSH_.xml"
INFO:cimpy.cimimport:START of parsing file "/dpsim/examples/Notebooks/Grids/logs/Crete 2030/20250429T1349Z___SV_.xml"
INFO:cimpy.cimimport:START of parsing file "/dpsim/examples/Notebooks/

In [12]:
name = 'cv_me_1'
reader = dpsim.CIMReader(name)
system = reader.loadCIM(50, files, dpsim.Domain.SP, dpsim.PhaseType.Single, dpsim.GeneratorType.PVNode) # PVNode only!
# Then reassign certain generators or loads to PQNode
#for comp in system.components:
#    if '_W3' in comp.name() or '_PO' in comp.name():
#        comp.generatorType = dpsim.GeneratorType.PQNode

system.render_to_file('logs/CIMImport.svg')

CIMContentHandler: Note: 0 out of 5524 tasks remain unresolved!


[10:03:45.780103 lne_90135_90136_1 warning] Zero value for Capacitance, setting default value of C=1e-12 [F]
[10:03:45.782098 lne_90131_90134_1 warning] Zero value for Capacitance, setting default value of C=1e-12 [F]
Uninitalized setPointVoltage for GeneratingUnit genstat_290531_W3. Using default value of 0
Uninitalized setPointVoltage for GeneratingUnit genstat_290131_W3. Using default value of 0
Uninitalized setPointVoltage for GeneratingUnit genstat_202101_PO. Using default value of 0
Uninitalized setPointVoltage for GeneratingUnit genstat_290231_W3. Using default value of 0
[10:03:45.788886 lne_61635_61933_1 warning] Zero value for Capacitance, setting default value of C=1e-12 [F]
[10:03:45.789037 lne_61635_61735_2 warning] Zero value for Capacitance, setting default value of C=1e-12 [F]
[10:03:45.791538 lne_90131_90133_1 warning] Zero value for Capacitance, setting default value of C=1e-12 [F]
[10:03:45.792529 lne_61735_61833_1 warning] Zero value for Capacitance, setting default

# IPTO Power profiles excel file

In [24]:
import pandas as pd
import os
import shutil

# Read power profiles excel file (sheets: active power and reactive power) with the third row as the header
# sheets = pd.read_excel('Profiles_Data/Power profiles data - Crete power system - 10 Jan.xlsx', sheet_name=['active power (MW)', 'reactive power (MVAR)'], header=2)
# sheets = pd.read_excel('Profiles_Data/Power profiles data - Crete power system - 10 June.xlsx', sheet_name=['active power (MW)', 'reactive power (MVAR)'], header=2)
sheets = pd.read_excel('logs/Power profiles data - Crete power system 2030 - 10 June with_PV.xlsx', sheet_name=['active power (MW)', 'reactive power (MVAR)'], header=2)

# Extract the datetime column from the first sheet (assuming the datetime column is the same across all sheets)
datetime_column = sheets['active power (MW)']['DD/MM/YYYY HH:MM'].values

# Find common columns names (component name)
common_cols = set(sheets['active power (MW)'].columns)

for sheet_name, df in sheets.items():
    common_cols &= set(df.columns)  # Keep only columns that appear in all sheets

common_cols.discard('DD/MM/YYYY HH:MM') # discard datetime column

common_cols= [s for s in common_cols if 'Unnamed' not in s] # Remove column with 'Unnamed'

## Save column's data to a CSV file named after the column
# Check if the folder exists and remove it (along with all its contents)
output_folder='profiles_dpsim'
if os.path.exists(output_folder):
    shutil.rmtree(output_folder)
    # print(f"Removed existing folder: {output_folder}")
os.makedirs(output_folder, exist_ok=True)

profiles_dict= {}

# Add the datetime column at the beginning of the dataframe
profiles_dict['DD/MM/YYYY HH:MM']= datetime_column

for col in common_cols:
    for sheet_name, df in sheets.items():
        if sheet_name == 'active power (MW)':
            profiles_dict[f'{col}_p'] = df[col]  # Add the active power profile data and extend column name with _p
        elif sheet_name == 'reactive power (MVAR)':
            profiles_dict[f'{col}_q'] = df[col]  # Add the reactive power profile data and extend column name with _p

profiles_df= pd.DataFrame.from_dict(profiles_dict)
profiles_df=profiles_df.dropna(how='all') # drop rows with all NaN values
profiles_df

,DD/MM/YYYY HH:MM,machine_291331_WO_MV_p,machine_291331_WO_MV_q,machine_290231_W3_MV_p,machine_290231_W3_MV_q,machine_291331_PV_MV_p,machine_291331_PV_MV_q,machine_91387_3_15.80kV_p,machine_91387_3_15.80kV_q,machine_291331_W3_MV_p,...,machine_92378_5_11.50kV_p,machine_92378_5_11.50kV_q,load_90431_1_MV_p,load_90431_1_MV_q,machine_291131_W3_MV_p,machine_291131_W3_MV_q,machine_92478_4_6.30kV_p,machine_92478_4_6.30kV_q,load_90931_1_MV_p,load_90931_1_MV_q
0,2024-10-06 00:00:00,15.304908,-1.218669,1.161416,-0.152869,0.0,0,-0.004884,-0.002442,15.304908,...,0,-0.197814,9.958457,8.928384,0,0,18.981685,5.520146,22.890633,5.844444
1,2024-10-06 00:01:00,15.176557,-1.127240,1.032479,-0.138217,0.0,0,-0.004884,-0.002442,15.176557,...,0,-0.197814,9.958459,8.857352,0,0,19.142856,5.593407,23.671729,5.865934
2,2024-10-06 00:02:00,15.823590,-1.127240,1.070574,-0.141148,0.0,0,-0.004884,-0.002442,15.823590,...,0,-0.197814,10.160794,8.768086,0,0,18.996337,5.593407,22.762750,5.756532
3,2024-10-06 00:03:00,15.482491,-0.947900,1.070574,-0.141148,0.0,0,-0.004884,-0.002442,15.482491,...,0,-0.197814,9.869190,8.906945,0,0,18.996337,5.520146,22.304510,5.845421
4,2024-10-06 00:04:00,15.252161,-1.127240,0.917216,-0.120635,0.0,0,-0.004884,-0.002442,15.252161,...,0,-0.197814,9.960441,8.468012,0,0,19.201465,5.520146,23.492840,5.559219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,2024-10-06 23:55:00,2.680146,-0.988535,0.673016,-0.063004,0.0,0,0.004884,0.002442,2.680146,...,0,-0.197814,12.217137,20.920659,0,0,18.952381,5.549450,23.165391,4.477900
1436,2024-10-06 23:56:00,2.766300,-1.076447,0.701343,-0.062027,0.0,0,0.004884,0.002442,2.766300,...,0,-0.197814,13.147491,20.762039,0,0,18.923076,5.549450,23.041330,4.579487
1437,2024-10-06 23:57:00,2.776850,-1.076447,0.592918,-0.051282,0.0,0,0.004884,0.002442,2.776850,...,0,-0.197814,11.929500,20.871143,0,0,18.791208,5.476191,23.007604,4.578510
1438,2024-10-06 23:58:00,2.599853,-0.979634,0.676923,-0.060073,0.0,0,0.004884,0.002442,2.599853,...,0,-0.197814,13.345867,20.551920,0,0,18.923076,5.476191,22.798064,4.580464


In [25]:
from pathlib import Path
import re

# Function to modify the profiles components names and make them similar to the simulation data components names
# Define a mapping of old substrings to new substrings
replace_map = {
    'lod': 'load',
    'sym': 'machine', # sym = synchronous machine --> has dynamics of rotor, exciter etc.
    'genstat': 'machine', # genstat = static generator --> has no dynamics
    'shntfix': 'fixed shunt',
    'shntswt': 'switched_shunt'
}

def modify_string(s, replace_map):
    # Replace firt substring with the mapped substring
    for old, new in replace_map.items():
        s = s.replace(old, new)  # Replace each part of the string
    return s

# List profiles component names
profiles_comp_names  = list(profiles_df.columns.values)

# List simulation component names
obj_list = system.list_idobjects()
simulation_comp_names= [k for k, v in obj_list.items() if  v== 'SP::Ph1::Load' or  v== 'SP::Ph1::Shunt' or  v== 'SP::Ph1::SynchronGenerator']

# old:new

sim_profile_map={}
columns_to_extract=['DD/MM/YYYY HH:MM']

# Iterate over simulation component names and check if they exist as a substring in profiles_comp_names
for sim_comp in simulation_comp_names:
    found = False
    start_substring= modify_string(sim_comp, replace_map)
    pattern = rf'^{re.escape(start_substring)}'
    # print(pattern)
    matches = [s for s in profiles_comp_names if re.match(pattern, s)]

    if matches:
        found = True
        for profile_comp in matches:
            if profile_comp.endswith('_p'):
                sim_profile_map[profile_comp]= f'{sim_comp}_p'
                columns_to_extract.append(profile_comp)
                print(f"Profile component '{profile_comp}' assigned to simulation component '{sim_comp}_p'")
            elif profile_comp.endswith('_q'):
                sim_profile_map[profile_comp]= f'{sim_comp}_q'
                columns_to_extract.append(profile_comp)
                print(f"Profile component '{profile_comp}' assigned to simulation component '{sim_comp}_q'")
    if not found:
        print(f"No profile component found for simulation component '{sim_comp}'")

columns_to_extract

# Create a new DataFrame with the selected columns
sim_profiles_df = profiles_df[columns_to_extract]
sim_profiles_df.rename(columns=sim_profile_map, inplace=True)
sim_profiles_df.to_csv(os.path.join('logs', 'sim_profiles.csv'), index=False)
sim_profiles_df.head()


No profile component found for simulation component 'Synchronous Machine'
Profile component 'machine_202101_PO_MV_p' assigned to simulation component 'genstat_202101_PO_p'
Profile component 'machine_202101_PO_MV_q' assigned to simulation component 'genstat_202101_PO_q'
Profile component 'machine_290131_W3_MV_p' assigned to simulation component 'genstat_290131_W3_p'
Profile component 'machine_290131_W3_MV_q' assigned to simulation component 'genstat_290131_W3_q'
Profile component 'machine_290231_W3_MV_p' assigned to simulation component 'genstat_290231_W3_p'
Profile component 'machine_290231_W3_MV_q' assigned to simulation component 'genstat_290231_W3_q'
Profile component 'machine_290333_W3_MV_p' assigned to simulation component 'genstat_290333_W3_p'
Profile component 'machine_290333_W3_MV_q' assigned to simulation component 'genstat_290333_W3_q'
Profile component 'machine_290531_W3_MV_p' assigned to simulation component 'genstat_290531_W3_p'
Profile component 'machine_290531_W3_MV_q' a

/tmp/ipykernel_68736/3482111938.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sim_profiles_df.rename(columns=sim_profile_map, inplace=True)


,DD/MM/YYYY HH:MM,genstat_202101_PO_p,genstat_202101_PO_q,genstat_290131_W3_p,genstat_290131_W3_q,genstat_290231_W3_p,genstat_290231_W3_q,genstat_290333_W3_p,genstat_290333_W3_q,genstat_290531_W3_p,...,sym_92378_5_p,sym_92378_5_q,sym_92587_4_p,sym_92587_4_q,sym_92687_3_p,sym_92687_3_q,sym_93278_3_p,sym_93278_3_q,sym_93378_4_p,sym_93378_4_q
0,2024-10-06 00:00:00,0.0,0,0.0,-1.349597,1.161416,-0.152869,2.044444,0.194139,11.114115,...,0,-0.197814,0,0.765917,0,-1.175896,29.799188,5.340281,29.789270,6.030305
1,2024-10-06 00:01:00,0.0,0,0.0,-1.349597,1.032479,-0.138217,2.471795,0.169231,10.804127,...,0,-0.197814,0,0.765917,0,-1.175896,29.799188,5.269249,29.719841,6.030305
2,2024-10-06 00:02:00,0.0,0,0.0,-1.349597,1.070574,-0.141148,2.483761,0.172161,10.269328,...,0,-0.197814,0,0.765917,0,-1.175896,29.799188,5.269249,29.880519,6.030305
3,2024-10-06 00:03:00,0.0,0,0.0,-1.349597,1.070574,-0.141148,2.456410,0.173626,11.385323,...,0,-0.197814,0,0.765917,0,-1.175896,29.769432,5.269249,29.769432,6.030305
4,2024-10-06 00:04:00,0.0,0,0.0,-1.349597,0.917216,-0.120635,2.545299,0.169231,10.717607,...,0,-0.197814,0,0.765917,0,-1.139264,29.769432,5.140232,29.860683,5.859249


# Setup DPsim simulation

In [34]:
sim = dpsim.Simulation(name, dpsim.LogLevel.info)
logger = dpsim.Logger(name)

for node in system.nodes:
    logger.log_attribute(node.name()+'.V', 'v', node)
    logger.log_attribute(node.name()+ '.S', 's', node)
    

for comp in system.components:
    if comp.__class__.__name__ =='PiLine':
        logger.log_attribute(comp.name() + '.Ibranch', 'current_vector', comp)
        logger.log_attribute(comp.name() + '.Pbranch', 'p_branch_vector', comp)
        logger.log_attribute(comp.name() + '.Qbranch', 'q_branch_vector', comp)
    if comp.__class__.__name__ =='Transformer':
        logger.log_attribute(comp.name() + '.Ibranch', 'current_vector', comp)
        logger.log_attribute(comp.name() + '.Pbranch', 'p_branch_vector', comp)
        logger.log_attribute(comp.name() + '.Qbranch', 'q_branch_vector', comp)

                     
sim.set_system(system)
sim.set_time_step(1)
# sim.set_final_time(1)
sim.set_final_time(1440)
sim.set_domain(dpsim.Domain.SP)
sim.set_solver(dpsim.Solver.NRP)
sim.set_solver_component_behaviour(dpsim.SolverBehaviour.Simulation)
sim.add_logger(logger)

# Read components and scenarios characteristics

In [32]:
# bus voltages
busbars_ratings=pd.read_excel('logs/ratings/busses.xlsx')
rated_volt_busses=dict(zip(busbars_ratings.iloc[:, 0], busbars_ratings.iloc[:, 1]))

sim_bus_voltages={'DD/MM/YYYY HH:MM': []}
for node in system.nodes:
    sim_bus_voltages[node.name()+ '_mag']=[]
    sim_bus_voltages[node.name()+ '_phase']=[]
    sim_bus_voltages[node.name()+ '_p']=[]
    sim_bus_voltages[node.name()+ '_q']=[]

busbars_ratings

,Name,Voltnom
0,202101 SOL_POW_ MV,20.0
1,2101 SOL_POW_HV,150.0
2,2102 SOL_POW_T1,150.0
3,2103 SOL_POW_T2,150.0
4,290031 CHANIA1 MV,33.0
...,...,...
109,94778 ATHER_F2,11.5
110,94779 ATHER_F3,11.5
111,94787 ATHER_F4,11.5
112,94788 ATHER_F5,11.5


In [31]:
lines_ratings=pd.read_excel('logs/ratings/lines.xlsx')
rated_curr_lines= {'DD/MM/YYYY HH:MM': []}
rated_curr_lines.update(dict(zip(lines_ratings.iloc[:, 0], lines_ratings.iloc[:, 1])))

sim_lines_loadings= {'DD/MM/YYYY HH:MM': []}
sim_lines_loadings.update({k: [] for k in rated_curr_lines if  k in system.list_idobjects()})
lines_ratings

,Name,Inom,From_bus,To_bus
0,lne_2101_2102_1,0.777498,2102 SOL_POW_T1,2101 SOL_POW_HV
1,lne_2101_2103_1,0.777498,2103 SOL_POW_T2,2101 SOL_POW_HV
2,lne_2102_90731_1,0.777498,90731 SITIA,2102 SOL_POW_T1
3,lne_2103_90431_1,0.777498,2103 SOL_POW_T2,90431 ATHERINO
4,lne_61631_61635_1,0.962251,61631 MOLAI1,61635 MOLAOI_TR1
...,...,...,...,...
64,lne_91841_91844_1,0.777498,91844 AER_TAP_2,91841 AER_IRAKL
65,lne_91844_92031_1,0.777498,92031 P.AMMOS,91844 AER_TAP_2
66,lne_92131_94631_1,0.777498,92131 PERAMA,94631 DAMASTA
67,lne_93133_93135_1,0.769993,93133 CHANIA1_SC1,93135 CHANIA1_UC1


In [30]:
transformers_ratings=pd.read_excel('logs/ratings/transformers.xlsx')
rated_power_transf= {'DD/MM/YYYY HH:MM': []}
rated_power_transf.update(dict(zip(transformers_ratings.iloc[:, 0], transformers_ratings.iloc[:, 1])))

sim_transf_loadings= {'DD/MM/YYYY HH:MM': []}
sim_transf_loadings.update({k: [] for k in rated_power_transf if  k in system.list_idobjects()})
transformers_ratings

,Name,Prated,From_bus,To_bus
0,trf_2101_202101_1,50.0,202101 SOL_POW_ MV,2101 SOL_POW_HV
1,trf_2101_202101_2,50.0,202101 SOL_POW_ MV,2101 SOL_POW_HV
2,trf_90031_290031_1,110.0,90031 CHANIA1,290031 CHANIA1 MV
3,trf_90031_290031_2,110.0,90031 CHANIA1,290031 CHANIA1 MV
4,trf_90031_90488_T4,31.5,90031 CHANIA1,90488 CHANI94
5,trf_90031_90588_T5,40.3,90031 CHANIA1,90588 CHANI95
6,trf_90031_90688_T6,52.0,90031 CHANIA1,90688 CHANI96
7,trf_90031_90788_T7,52.0,90031 CHANIA1,90788 CHANI97
8,trf_90031_90888_T8,52.0,90031 CHANIA1,90888 CHANI98
9,trf_90032_290032_1,50.0,290032 CHANIA2 MV,90032 CHANIA2


In [14]:
res_factors=pd.read_excel('logs/ratings/factors.xlsx')
res_factors

,Name,bus,type,scenario1,scenario2,scenario3,scenario4,scenario5
0,genstat_202101_PO,202101 SOL_POW_ MV,PO,1,1,1,1.3,1.300000
1,genstat_290032_PV,290032 CHANIA2 MV,PV,1,1,1,1.3,0.796813
2,genstat_290131_PV,290131 RETHIMNO MV,PV,1,1,1,1.3,1.481481
3,genstat_290231_PV,290231 LNPERAMA MV,PV,1,1,1,1.3,4.347826
4,genstat_290331_PV,290331 HR1_MT,PV,1,1,1,1.3,12.500000
5,genstat_290332_PV,290332 IRAKLIO2 MV,PV,1,1,1,1.3,1.052632
6,genstat_290333_PV,290333 IRAKLIO3 MV,PV,1,1,1,1.3,1.990050
7,genstat_290531_PV,290531 MIRES MV,PV,1,1,1,1.3,0.623053
8,genstat_290631_PV,290631 IERAPETR MV,PV,1,1,1,1.3,2.061856
9,genstat_290731_PV,290731 SITIA MV,PV,1,1,1,1.3,3.333333


## Function to assign PQ values in DPsim

In [35]:
def pq_assign_dpsim(sim_profiles_df, timestamp, gen_factor, load_factor, wind_factor, pv_factor, scenario):

    kw_w=1e3
    mw_w=1e6
    
    # List simulation component names
    obj_list = system.list_idobjects()
    simulation_comp_names= [k for k, v in obj_list.items() if  v== 'SP::Ph1::Load' or  v== 'SP::Ph1::Shunt' or  v== 'SP::Ph1::SynchronGenerator']
    
    # IMPORTANT CONVENTION FOR POWERFLOW --> positive sign consumption, negative sign production

    # 1e8 ?? --> maybe Sbase = 100 MVA for normalizing in p.u. --> then also lod and shunt have to be normalized

    for comp in simulation_comp_names:

        if f'{comp}_p' in sim_profiles_df.columns.values:
            P_set = sim_profiles_df[sim_profiles_df['DD/MM/YYYY HH:MM'] == timestamp][f'{comp}_p'].values[0]
                
            if 'lod' in comp: # load
                try:
                    sim.get_idobj_attr(comp, 'P').set(P_set*mw_w*load_factor) # + --> consumption
                except Exception as e:
                    print(f"[Warning] Could not set Q for {comp}: {e}")
            elif 'shnt' in comp: # shunt
                try:  
                    sim.get_idobj_attr(comp, 'P').set(P_set*mw_w*load_factor) # + --> consumption
                except Exception as e:
                    print(f"[Warning] Could not set Q for {comp}: {e}")
            elif '_W3' in comp: # wind
                wind_factor= res_factors[res_factors['Name']==comp][scenario].item()
                try:
                    sim.get_idobj_attr(comp, 'P').set(-P_set*mw_w*wind_factor) # why 1e8 ?!
                except Exception as e:
                    print(f"[Warning] Could not set Q for {comp}: {e}")   
            elif '_WO' in comp: # wind
                wind_factor= res_factors[res_factors['Name']==comp][scenario].item()
                try:
                    sim.get_idobj_attr(comp, 'P').set(-P_set*mw_w*wind_factor) # why 1e8 ?!
                except Exception as e:
                    print(f"[Warning] Could not set Q for {comp}: {e}")
            elif '_PV' in comp: # photovoltaic
                pv_factor= res_factors[res_factors['Name']==comp][scenario].item()
                try:
                    sim.get_idobj_attr(comp, 'P').set(-P_set*mw_w*pv_factor) # why 1e8 ?!
                except Exception as e:
                    print(f"[Warning] Could not set Q for {comp}: {e}")   
            elif '_PO' in comp: # photovoltaic
                pv_factor= res_factors[res_factors['Name']==comp][scenario].item()
                #sim.get_idobj_attr(comp, 'P').set(-P_set*mw_w*pv_factor)  # why 1e8 ?!
                try:
                    sim.get_idobj_attr(comp, 'Q').set(-Q_set * mw_w * pv_factor)
                except Exception as e:
                    print(f"[Warning] Could not set Q for {comp}: {e}")

            elif 'sym' in comp: # synchronous machine
                try:
                    sim.get_idobj_attr(comp, 'P_set_pu').set(P_set*mw_w*gen_factor/1e8) 
                except Exception as e:
                    print(f"[Warning] Could not set Q for {comp}: {e}")    
        elif f'{comp}_q' in sim_profiles_df.columns.values:
            Q_set = sim_profiles_df[sim_profiles_df['DD/MM/YYYY HH:MM'] == timestamp][f'{comp}_q'].values[0]
                
            if 'lod' in comp:
                sim.get_idobj_attr(comp, 'Q').set(Q_set*mw_w*load_factor)
            elif 'shnt' in comp:
                sim.get_idobj_attr(comp, 'Q').set(Q_set*mw_w*load_factor)
            elif '_W3' in comp:
                wind_factor= res_factors[res_factors['Name']==comp][scenario].item()
                try:
                    sim.get_idobj_attr(comp, 'Q').set(-Q_set*mw_w*wind_factor)
                except Exception as e:
                    print(f"[Warning] Could not set Q for {comp}: {e}")   
            elif '_WO' in comp:
                wind_factor= res_factors[res_factors['Name']==comp][scenario].item()
                sim.get_idobj_attr(comp, 'Q').set(-Q_set*mw_w*wind_factor)
            elif '_PV' in comp:
                pv_factor= res_factors[res_factors['Name']==comp][scenario].item()
                sim.get_idobj_attr(comp, 'Q').set(-Q_set*mw_w*pv_factor)
            elif '_PO' in comp:
                pv_factor= res_factors[res_factors['Name']==comp][scenario].item()
                try:
                    sim.get_idobj_attr(comp, 'Q').set(-Q_set * mw_w * pv_factor)
                except Exception as e:
                    print(f"[Warning] Could not set Q for {comp}: {e}")
                #sim.get_idobj_attr(comp, 'Q').set(-Q_set*mw_w*pv_factor)
            elif 'sym' in comp: 
                sim.get_idobj_attr(comp, 'Q_set_pu').set(Q_set*mw_w*gen_factor/1e8) # 1e8 because of setpoint pu

        else:
            continue


In [17]:
import math, cmath

def run_dpsim_simulation(gen_factor, load_factor, wind_factor, pv_factor, scenario):
    sim.start()
        
    # Loop over the timeseries
    for index, row in sim_profiles_df.iterrows():
        timestamp = row['DD/MM/YYYY HH:MM']
        pq_assign_dpsim(sim_profiles_df, timestamp, gen_factor, load_factor, wind_factor, pv_factor, scenario)
        sim.next()
        
        # lines loadings
        sim_lines_loadings['DD/MM/YYYY HH:MM'].append(timestamp)
        for comp in system.components:
            if comp.__class__.__name__ =='PiLine':
                # line current from simulation
                sim_curr_mag= sim.get_idobj_attr(comp.name(), 'current_vector').derive_coeff(0,0).derive_mag().get()
                # sim_curr_mag= sim.get_idobj_attr(comp.name(), 'current_vector').derive_coeff(1,0).derive_mag().get()

                # rated line current from dict
                rated_curr_mag= rated_curr_lines[comp.name()]*1e3 # kA
                # calculate line loading
                line_loading_percent= 100*sim_curr_mag/rated_curr_mag
                sim_lines_loadings[comp.name()].append(line_loading_percent)
                # print(line_loading_percent)
    
        # Transformer loadings
        sim_transf_loadings['DD/MM/YYYY HH:MM'].append(timestamp)
        for comp in system.components:
            if comp.__class__.__name__ =='Transformer':
                # transformer power from simulation
                sim_power= sim.get_idobj_attr(comp.name(), 'p_inj').get()
                # rated Transformer power from dict
                rated_power= rated_power_transf[comp.name()]*1e6 # MW
                # calculate transformer loading
                transf_loading_percent= 100*abs(sim_power/rated_power)
                sim_transf_loadings[comp.name()].append(transf_loading_percent)
                # print(transf_loading_percent)   
        
        # bus voltages
        sim_bus_voltages['DD/MM/YYYY HH:MM'].append(timestamp)
        for node in system.nodes:
                # bus voltages from simulation
                sim_volt_mag= sim.get_idobj_attr(node.name(), 'v').derive_coeff(0,0).derive_mag().get() / (1e3* rated_volt_busses[node.name()])
                sim_volt_phase= sim.get_idobj_attr(node.name(), 'v').derive_coeff(0,0).derive_phase().get()
                sim_volt_p_mw= sim.get_idobj_attr(node.name(), 's').derive_coeff(0,0).derive_real().get()/1e6
                sim_volt_q_mvar= sim.get_idobj_attr(node.name(), 's').derive_coeff(0,0).derive_imag().get()/1e6
                sim_bus_voltages[node.name()+ '_mag'].append(sim_volt_mag)
                sim_bus_voltages[node.name()+ '_phase'].append(math.degrees(sim_volt_phase))
                sim_bus_voltages[node.name()+ '_p'].append(sim_volt_p_mw)
                sim_bus_voltages[node.name()+ '_q'].append(sim_volt_q_mvar)
                
    
    sim.stop()


    ############### TODO EXTRACT RESULTS FROM SIMULATION LOGS ###################
    #res_folder='logs/Results/'+ scenario
    #if not os.path.exists(res_folder):
    #    os.makedirs(res_folder, exist_ok=True)
    
    #lines_loadings_df= pd.DataFrame.from_dict(sim_lines_loadings)
    #lines_loadings_df.to_csv(os.path.join('Results', scenario, 'lines_loading_' + scenario + '_' + str(gen_factor) + 'Gen_' + str(load_factor) + 'Load_' + str(wind_factor) + 'Wind_' + str(pv_factor) +'PV.csv'), index=False)
    
    #transf_loadings_df= pd.DataFrame.from_dict(sim_transf_loadings)
    #transf_loadings_df.to_csv(os.path.join('Results', scenario, 'transformers_loading_' + scenario + '_' + str(gen_factor) + 'Gen_' + str(load_factor) + 'Load_' + str(wind_factor) + 'Wind_' + str(pv_factor) +'PV.csv'), index=False)
    
    #bus_voltages_df= pd.DataFrame.from_dict(sim_bus_voltages)
    #bus_voltages_df.to_csv(os.path.join('Results', scenario, 'bus_voltages_' + scenario + '_' + str(gen_factor) + 'Gen_' + str(load_factor) + 'Load_' + str(wind_factor) + 'Wind_' + str(pv_factor) +'PV.csv'), index=False)

## Run DPsim simulation Stepwise

In [ ]:
# gen_factor, load_factor, wind_factor, pv_factor

# Scenarios
#run_dpsim_simulation(1, 1, 1, 1, 'scenario0')

# Scenario 1: 100% RES, 90% CG
#run_dpsim_simulation(0.9, 1, 1, 1, 'scenario1')

# Scenario 2: 100% RES, 50% CG
# run_dpsim_simulation(0.5, 1, 1, 1, 'scenario2')

# Scenario 3: 100% RES, 0% CG
# run_dpsim_simulation(0, 1, 1, 1, 'scenario3')

# Scenario 4: 150% wind, 130% PV, 0% CG
run_dpsim_simulation(0, 1, 1.5, 1.3, 'scenario4')

# Scenario 5: considering single factors for each unit
# run_dpsim_simulation(0, 1, 2.5, 2, 'scenario5')

## Read DPsim results

In [ ]:
dpsim_result_file = 'logs/' + name + '.csv'
ts_dpsim = read_timeseries_csv(dpsim_result_file)

In [ ]:
# total_load_p=0
# total_load_q=0
# total_gen_p=0
# total_gen_q=0

# # This is hardcoded depending on the names of the compoenents in the network model "Crete 2030 Slack"
# for comp in system.components:
#     if 'lod' in comp.name():
#         total_load_p= total_load_p + sim.get_idobj_attr(comp.name(), 'P').get()
#         total_load_q= total_load_q + sim.get_idobj_attr(comp.name(), 'Q').get()
#     elif 'sym' in comp.name():
#         total_gen_p= total_gen_p + sim.get_idobj_attr(comp.name(), 'P_set').get()
#         total_gen_q= total_gen_q + sim.get_idobj_attr(comp.name(), 'Q_set').get()
#     elif 'genstat' in comp.name():
#         total_gen_p= total_gen_p - sim.get_idobj_attr(comp.name(), 'P').get()
#         total_gen_q= total_gen_q - sim.get_idobj_attr(comp.name(), 'Q').get()
#     elif 'Synchronous Machine' in comp.name():
#         total_gen_p= total_gen_p - sim.get_idobj_attr(comp.name(), 'P_set').get()
#         total_gen_q= total_gen_q - sim.get_idobj_attr(comp.name(), 'Q_set').get()

# df_dpsim_summary_results = pd.DataFrame(data={'Generation P[MW]': [total_gen_p/1e6], 'Generation Q[MVAR]': [total_gen_q/1e6], 'Load P[MW]': [total_load_p/1e6], 'Load Q[MVAR]': [total_load_q/1e6]})
# pd.set_option('display.max_rows', None)
# pd.options.display.float_format = '{:.3f}'.format
# df_dpsim_summary_results

In [ ]:
# This shows the results of the first time step
# The results of all time steps are saved in the folder "Results"
import math, cmath

dpsim_bus_results = []

for node in system.nodes:
    row= {'name': node.name(),
          'vm [kV]': cmath.polar(ts_dpsim[node.name()+'.V'].values[0])[0]/1e3,
          'va [deg]': math.degrees(cmath.polar(ts_dpsim[node.name()+'.V'].values[0])[1]),
          'p_inj [MW]':ts_dpsim[node.name()+'.S'].values[0].real/1e6,
          'q_inj [MVAR]':ts_dpsim[node.name()+'.S'].values[0].imag/1e6}
    dpsim_bus_results.append(row)

        

df_dpsim_bus_results = pd.DataFrame(dpsim_bus_results, columns=['name', 'vm [kV]', 'va [deg]', 'p_inj [MW]', 'q_inj [MVAR]'])
pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:.3f}'.format
df_dpsim_bus_results

In [ ]:
# This shows the results of the first time step
# The results of all time steps are saved in the folder "Results"
import math, cmath

dpsim_branch_results = []

for comp in system.components:
    if comp.__class__.__name__ =='PiLine' or comp.__class__.__name__ =='Transformer':
        row= {'name': comp.name(),
              'i_0 [kA]': abs(ts_dpsim[comp.name()+'.Ibranch_0'].values[0]/1e3),  
              'p_0 [MW]': ts_dpsim[comp.name()+'.Pbranch_0'].values[0]/1e6,
              'q_0 [MVAR]': ts_dpsim[comp.name()+'.Qbranch_0'].values[0]/1e6,
              'i_1 [kA]': abs(ts_dpsim[comp.name()+'.Ibranch_1'].values[0]/1e3),
              'p_1 [MW]': ts_dpsim[comp.name()+'.Pbranch_1'].values[0]/1e6,
              'q_1 [MVAR]': ts_dpsim[comp.name()+'.Qbranch_1'].values[0]/1e6}
        dpsim_branch_results.append(row)

df_dpsim_branch_results = pd.DataFrame(dpsim_branch_results, columns=['name', 'i_0 [kA]', 'p_0 [MW]', 'q_0 [MVAR]', 'i_1 [kA]', 'p_1 [MW]', 'q_1 [MVAR]'])
pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:.3f}'.format
df_dpsim_branch_results